# 19 — Channel factorial: belief accuracy vs closed-loop profit

One **joint shard** per `(seed, ObsChannels)` runs a single closed-loop `act()` episode and records **mean-f MAE**, **8-bin distribution MAE**, and **profit** from the same scored days.

Canonical grid: `(upc|gsin) × (waste on|off) × (none|pack_date|temperature_history)` — 12 cells.

Budget target: ~20 min wall / 2 CPU-hr on Modal (probe → plan → run).

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / "src" / "blueberries_voi").is_dir() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "experiments" / "data"
FIG_DIR = REPO_ROOT / "figures" / "channel_joint"
OUT_JSON = DATA_DIR / "nb19_joint_rows.json"

_wheel_dir = REPO_ROOT / "dist" / "wheel"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)

from blueberries_voi.experiments.batch_budget import assert_within_budget, plan_channel_joint_budget
from blueberries_voi.experiments.channel_factorial_viz import save_nb19_figures
from blueberries_voi.experiments.channel_joint import (
    all_obs_channels_product,
    channel_joint_job_grid,
    run_seed_channel_joint,
)
from blueberries_voi.experiments.modal_dispatch import run_batch
from blueberries_voi.filter.types import channels_for_preset

BATCH_MODE: Literal["modal", "local"] = "modal"
SMOKE = False
ACCURACY_METRIC: Literal["mean_f", "distribution"] = "mean_f"

PROBE_SEED = 42
PROBE_CHANNEL = channels_for_preset("P0")
CHANNELS = all_obs_channels_product()
CANDIDATE_SEEDS = (42, 7, 99, 101, 2024, 31415)

## Probe — one shard wall time

In [ ]:
probe_t0 = time.perf_counter()
probe_row = run_seed_channel_joint(
    PROBE_SEED,
    PROBE_CHANNEL,
    n_burn=2,
    n_score=10,
)
probe_elapsed_s = time.perf_counter() - probe_t0
print(f"probe elapsed_s={probe_elapsed_s:.1f} mae_f={probe_row['mae_f']:.4f} profit={probe_row['profit']:.2f}")

## Plan — greedy seeds then bump `n_score` under Modal budget

In [ ]:
plan = plan_channel_joint_budget(probe_elapsed_s, max_seeds=len(CANDIDATE_SEEDS))
assert_within_budget(plan)
SEEDS = tuple(CANDIDATE_SEEDS[:plan.n_seeds])
N_BURN = plan.n_burn
N_SCORE = plan.n_score
print(plan.as_dict())
print(f"grid={len(SEEDS)} seeds × {len(CHANNELS)} channels = {len(SEEDS) * len(CHANNELS)} shards")

## Run — Modal or local batch

In [ ]:
run_t0 = time.perf_counter()
rows = run_batch(
    "channel_joint",
    BATCH_MODE,
    smoke=SMOKE,
    seeds=SEEDS,
    channels=CHANNELS,
    n_burn=N_BURN,
    n_score=N_SCORE,
    out_path=OUT_JSON,
)
run_wall_s = time.perf_counter() - run_t0
df = pd.DataFrame(rows)
print(f"run wall_s={run_wall_s:.1f} rows={len(df)}")
df.head()

## Audit — shard coverage and CPU estimate

In [ ]:
expected = len(channel_joint_job_grid(SEEDS, CHANNELS))
assert len(rows) == expected, (len(rows), expected)
est_cpu_hr = (len(rows) * probe_elapsed_s) / 3600.0
audit = {
    "shards": len(rows),
    "seeds": list(SEEDS),
    "n_score": N_SCORE,
    "n_burn": N_BURN,
    "probe_elapsed_s": probe_elapsed_s,
    "run_wall_s": run_wall_s,
    "est_cpu_hr": est_cpu_hr,
    "accuracy_metric": ACCURACY_METRIC,
}
print(json.dumps(audit, indent=2))
agg = df.groupby(["code_type", "waste", "delivery"], observed=True).agg(
    mae_f=("mae_f", "mean"),
    mae_dist=("mae_dist", "mean"),
    profit=("profit", "mean"),
)
agg

## Plots

In [ ]:
acc_col = "mae_f" if ACCURACY_METRIC == "mean_f" else "mae_dist"
written = save_nb19_figures(rows, FIG_DIR, accuracy_column=acc_col)
for path in written:
    print(path.relative_to(REPO_ROOT))